In [1]:
from winogender_contextuality.utils import *
from winogender_contextuality.modeling.contextuality import *
from winogender_contextuality.config import * 
from winogender_contextuality.modeling.prompting import *
import ast
#from winogender_contextuality.modeling.meta_prompting import *
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

2026-03-15 14:41:17.236 | INFO     | winogender_contextuality.config:<module>:13 - PROJ_ROOT path is: /home/sagar/winogender_contextuality
2026-03-15 14:41:17.238 | INFO     | winogender_contextuality.config:<module>:17 - DATA_ROOT path is: /data_users1/sagar/winogender_contextuality


# Constants

In [2]:
model_names = ['gpt', 'phi', 'gemma', 'llama1b', 'llama8b', 'qwen']

### Experiment Measurements

In [3]:
gpt_wp = load_ndjson(INTERIM_DATA_DIR / "one_pronoun_measurements_gpt-oss-20b_0.5_wp_k40.ndjson")
max_gpt = max([x["index"] for x in gpt_wp])
f"{len(gpt_wp)} Measuremenents"

'579200 Measuremenents'

In [4]:
phi_wp = load_ndjson(INTERIM_DATA_DIR / "one_pronoun_measurements_phi-4_0.5_wp_k40.ndjson")
max_phi = max([x["index"] for x in phi_wp])
f"{len(phi_wp)} Measuremenents"

'507354 Measuremenents'

In [5]:
gemma_wp = load_ndjson(INTERIM_DATA_DIR / "one_pronoun_measurements_gemma-3-12b-it_0.5_wp_k40.ndjson")
max_gemma = max([x["index"] for x in gemma_wp])
f"{len(gemma_wp)} Measuremenents"

'426124 Measuremenents'

In [6]:
llama1b_wp = load_ndjson(INTERIM_DATA_DIR / "one_pronoun_measurements_Llama-3.2-1B-Instruct_0.5_1331141225.ndjson")
max_llama1b = max([x["index"] for x in llama1b_wp])
f"{len(llama1b_wp)} Measuremenents"

'1158400 Measuremenents'

In [7]:
llama8b_wp = load_ndjson(INTERIM_DATA_DIR / "one_pronoun_measurements_Llama-3.1-8B-Instruct_0.5_wp_k40.ndjson")
max_llama8b = max([x["index"] for x in llama8b_wp])
f"{len(llama8b_wp)} Measuremenents"

'724000 Measuremenents'

In [8]:
qwen_wp = load_ndjson(INTERIM_DATA_DIR / "one_pronoun_measurements_Qwen2.5-7B-Instruct_0.5_wp_k40.ndjson")
max_qwen = max([x["index"] for x in qwen_wp])
f"{len(qwen_wp)} Measuremenents"

'694400 Measuremenents'

In [9]:
results_list = [gpt_wp, phi_wp, gemma_wp, llama1b_wp, llama8b_wp, qwen_wp]

### Metaprompting Measurements

In [10]:
questions = ['anaphora', 'pos', 'other_gender']

In [11]:
gpt_meta = load_ndjson(INTERIM_DATA_DIR / 
                       "metaprompting_gpt-oss-20b_0.5_1037130326.ndjson")

FileNotFoundError: [Errno 2] No such file or directory: '/data_users1/sagar/winogender_contextuality/data/interim/metaprompting_gpt-oss-20b_0.5_1037130326.ndjson'

In [ ]:
phi_meta = load_ndjson(INTERIM_DATA_DIR / 
                       "metaprompting_phi-4_0.5_0856130326.ndjson")

In [ ]:
gemma_meta = load_ndjson(INTERIM_DATA_DIR / 
                         "metaprompting_gemma-3-12b-it_0.5_1511150326.ndjson")

In [ ]:
llama1b_meta = load_ndjson(INTERIM_DATA_DIR / 
                           "metaprompting_Llama-3.2-1B-Instruct_0.5_1907110326.ndjson")

In [ ]:
llama8b_meta = load_ndjson(INTERIM_DATA_DIR / 
                           "metaprompting_Llama-3.1-8B-Instruct_0.5_1858110326.ndjson")

In [ ]:
qwen_meta = load_ndjson(INTERIM_DATA_DIR / 
                        "metaprompting_Qwen2.5-7B-Instruct_0.5_1106130326.ndjson" )

In [ ]:
models_meta = [gpt_meta, phi_meta, gemma_meta, llama1b_meta, llama8b_meta, qwen_meta]

# Functions

In [15]:
def get_meta_index(idx: int,
                   lst: list[dict]):

    """
    :param idx: pair index
    :param lst: a list of MetaQA objects or equivalent dictionaries 

    Filters list of responses by index.
    """

    return [x for x in lst if x['index'] == idx]
    
def get_meta_question(q: str,
                      lst: list[dict]):

    """
    :param q: string key of the question
    :param lst: a list of MetaQA objects or equivalent dictionaries 

    Filters list of responses by question.
    """

    return [x for x in lst if x['question'] == q]

In [16]:
def get_meta_scores(lst: list[dict],
                   questions: list[str] = ['anaphora', 'pos', 'other_gender']):
    
    dic = {idx: {} for idx in range(60)}


    for idx in range(60):
        for q in questions:
            responses = get_meta_question(q, get_meta_index(idx, lst))
            n_correct = len([r for r in responses if r['response'] == r['answer'].replace("the ", "").replace("a ", "").strip().lower()])
            dic[idx][q] = n_correct/8 # as a fraction of best possible (including errors)

    return dic
        

In [17]:
def get_average_meta(dic: dict):

    response_arr = []
    for idx, dic in dic.items():
        response_arr.append(list(dic.values()))
    response_arr = np.array(response_arr)

    return np.mean(response_arr, axis=0)

In [18]:
def plot_response_dict(response_d):
    # Extract data
    indices = list(response_d.keys())
    categories = list(next(iter(response_d.values())).keys())
    colors = plt.cm.viridis(np.linspace(0, 1, len(categories)))  # color per category

    # Create plot
    plt.figure(figsize=(8, 5))

    for cat, color in zip(categories, colors):
        vals = [response_d[i][cat] for i in indices]
        plt.scatter(indices, vals, color=color, label=cat, s=50, alpha=0.7)

    # Labels and style
    plt.xlabel('Index')
    plt.ylabel('Score')
    plt.title('Response Dictionary Scatter Plot')
    plt.xticks(indices)
    plt.ylim(0, 1)
    plt.legend(title='Category')
    plt.grid(True, linestyle='--', alpha=0.6)

In [19]:
def get_valid_unprimed_counts(data: list[Measurement],
                              max_index: int = 181
                             ) -> dict:
    unprimed_counts = {sent_idx: {} for sent_idx in range(2 * max_gpt)}
    for idx in tqdm(range(max_index)): 
    
        # [m,f] option order 
        fwd_ct_mf = len(get_pnoun_order(0, get_single_sentences(get_sent_order([0,1], get_index(idx, data))))) 
        bwd_ct_mf = len(get_pnoun_order(0, get_single_sentences(get_sent_order([1,0], get_index(idx, data))))) 
    
        fwd_ct_fm = len(get_pnoun_order(1, get_single_sentences(get_sent_order([0,1], get_index(idx, data))))) 
        bwd_ct_fm = len(get_pnoun_order(1, get_single_sentences(get_sent_order([1,0], get_index(idx, data))))) 
    
        unprimed_counts[2*idx]["mf"] = fwd_ct_mf
        unprimed_counts[2*idx + 1]["mf"] = fwd_ct_mf
        unprimed_counts[2*idx]["fm"] = fwd_ct_fm
        unprimed_counts[2*idx + 1]["fm"] = fwd_ct_fm
    return unprimed_counts

In [20]:
def get_valid_primed_counts(data: list[Measurement],
                            pnoun: int,
                            max_index: int = 181
    ) -> dict:

    """
    :param data:
    :param pnoun: 0 for masc, 1 for fem
    :param max_index: 
    """
    
    p_primed_counts = {sent_idx: {} for sent_idx in range(2 * max_gpt)}
    for idx in tqdm(range(max_index)): 
    
        # [m,f] option order 
        fwd_ct_mf = len(get_pnoun_order(0, get_filled_pnoun(pnoun, get_sent_order([0,1], get_index(idx, data))))) 
        bwd_ct_mf = len(get_pnoun_order(0, get_filled_pnoun(pnoun, get_sent_order([1,0], get_index(idx, data))))) 
    
        fwd_ct_fm = len(get_pnoun_order(1, get_filled_pnoun(pnoun, get_sent_order([0,1], get_index(idx, data)))))
        bwd_ct_fm = len(get_pnoun_order(1, get_filled_pnoun(pnoun, get_sent_order([1,0], get_index(idx, data))))) 
    
        p_primed_counts[2*idx]["mf"] = fwd_ct_mf
        p_primed_counts[2*idx + 1]["mf"] = fwd_ct_mf
        p_primed_counts[2*idx]["fm"] = fwd_ct_fm
        p_primed_counts[2*idx + 1]["fm"] = fwd_ct_fm
        
    return p_primed_counts

# Counting Valid Responses

In [59]:
unprimed_model_counts = []
for model in results_list:
    unprimed_model_counts.append(get_valid_unprimed_counts(model))

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

In [60]:
masc_primed_model_counts = []
for model in results_list:
    masc_primed_model_counts.append(get_valid_primed_counts(model, 0))

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

In [61]:
fem_primed_model_counts = []
for model in results_list:
    fem_primed_model_counts.append(get_valid_primed_counts(model, 1))

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

In [69]:
un_m_count = 0
un_f_count = 0

masc_m_count = 0
masc_f_count = 0

fem_m_count = 0
fem_f_count = 0

for idx, d in unprimed_model_counts[-1].items():
    un_m_count += d['mf']
    un_f_count += d['fm']

for idx, d in masc_primed_model_counts[-1].items():
    masc_m_count += d['mf']
    masc_f_count += d['fm']

for idx, d in fem_primed_model_counts[-1].items():
    fem_m_count += d['mf']
    fem_f_count += d['fm']


un_m_count, un_f_count, masc_m_count*2/un_m_count, masc_f_count*2/un_m_count, fem_m_count*2/un_m_count, fem_f_count*2/un_m_count

(172400, 172400, 1.0, 1.0, 1.0, 1.0)

In [70]:
ratio_un_m = []
ratio_un_f = []
ratio_masc_m = []
ratio_masc_f = []
ratio_fem_m = []
ratio_fem_f = []

for i,model in enumerate(results_list):
    total_measurements_per_setting = int(.25*len(model))
    
    un_m_count = 0
    un_f_count = 0
    
    masc_m_count = 0
    masc_f_count = 0
    
    fem_m_count = 0
    fem_f_count = 0
    
    for idx, d in unprimed_model_counts[i].items():
        un_m_count += d['mf']
        un_f_count += d['fm']
    
    for idx, d in masc_primed_model_counts[i].items():
        masc_m_count += d['mf']
        masc_f_count += d['fm']
    
    for idx, d in fem_primed_model_counts[i].items():
        fem_m_count += d['mf']
        fem_f_count += d['fm']
    
    ratio_un_m.append(f"{un_m_count}/{total_measurements_per_setting}")
    ratio_un_f.append(f"{un_f_count}/{total_measurements_per_setting}")
    ratio_masc_m.append(f"{masc_m_count}/{int(.5*total_measurements_per_setting)}")
    ratio_masc_f.append(f"{masc_f_count}/{int(.5*total_measurements_per_setting)}")
    ratio_fem_m.append(f"{fem_m_count}/{int(.5*total_measurements_per_setting)}")
    ratio_fem_f.append(f"{fem_f_count}/{int(.5*total_measurements_per_setting)}")

In [71]:
counts_df = pd.DataFrame()

counts_df['name'] = model_names
counts_df['Unprimed MF Order'] = ratio_un_m 
counts_df['Unprimed FM Order'] = ratio_un_f
counts_df['Masc Primed MF Order'] = ratio_masc_m
counts_df['Masc Primed FM Order'] = ratio_masc_f
counts_df['Fem Primed MF Order'] = ratio_fem_m
counts_df['Fem Primed FM Order'] = ratio_fem_f

In [72]:
counts_df

,name,Unprimed MF Order,Unprimed FM Order,Masc Primed MF Order,Masc Primed FM Order,Fem Primed MF Order,Fem Primed FM Order
0,gpt,144000/144800,144000/144800,72000/72400,72000/72400,72000/72400,72000/72400
1,phi,126566/126838,126460/126838,63124/63419,63056/63419,63116/63419,62978/63419
2,gemma,106770/106531,106400/106531,53300/53265,53196/53265,53276/53265,53038/53265
3,llama1b,270206/289600,275454/289600,107566/144800,96280/144800,108138/144800,100888/144800
4,llama8b,180000/181000,180000/181000,90000/90500,90000/90500,90000/90500,90000/90500
5,qwen,172400/173600,172400/173600,86200/86800,86200/86800,86200/86800,86200/86800


# Counting Correct Responses

# Metaprompting Scores 

In [ ]:
for model, model_name in zip(models_meta, model_names):
    scores = get_meta_scores(model)
    average = get_average_meta(scores)

    print(f"{model_name}: {average}")